In [15]:
# 1 Configuracion

import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.evaluation.stats import paired_bootstrap

L = ROOT / "results" / "large"
RES = ROOT / "results" / "metrics"

resumen = pd.read_csv(next((L / "resumen").glob("*.csv")))
tiempos = pd.read_csv(next((L / "tiempos").glob("*.csv")))
m = pd.read_parquet(next((L / "por_impresion").glob("*.parquet")))
print(f"Impresiones evaluadas en MIND-large: {len(m):,}")

Impresiones evaluadas en MIND-large: 376,471


In [16]:
# 2 resumen MIND-large
nombres = {"hibrido": "Híbrido (CTR + embeddings)", "ctr": "Baseline: CTR reciente 24 h",
           "emb_resp": "Embeddings MiniLM", "aleatorio": "Azar"}
resumen["modelo"] = resumen["modelo"].map(nombres)
resumen.sort_values("auc", ascending=False).round(4)

,modelo,auc,mrr,ndcg5,ndcg10
3,Híbrido (CTR + embeddings),0.6828,0.3355,0.3721,0.4340
1,Baseline: CTR reciente 24 h,0.6534,0.3129,0.3418,0.4038
2,Embeddings MiniLM,0.6401,0.3085,0.3369,0.3969
0,Azar,0.4993,0.2184,0.2229,0.2858


In [17]:
# Nombres comunes para ambas tablas (claves originales de cada archivo)
nombres = {
    "hibrido": "Híbrido (CTR + embeddings)",
    "ctr": "Baseline: CTR reciente 24 h", "baseline_ctr": "Baseline: CTR reciente 24 h",
    "emb_resp": "Embeddings MiniLM", "embeddings": "Embeddings MiniLM",
    "aleatorio": "Azar",
}
METS = ["auc", "mrr", "ndcg5", "ndcg10"]

large = pd.read_csv(next((L / "resumen").glob("*.csv")))
small = pd.read_csv(RES / "07_test_final.csv")

large["modelo"] = large["modelo"].replace(nombres)
small["modelo"] = small["modelo"].replace(nombres)
small = small[small["modelo"].isin(large["modelo"])]

comp = (small[["modelo"] + METS]
        .merge(large[["modelo"] + METS], on="modelo", suffixes=("_small", "_large"))
        .sort_values("auc_large", ascending=False)
        .reset_index(drop=True))
comp.round(4)

,modelo,auc_small,mrr_small,ndcg5_small,ndcg10_small,auc_large,mrr_large,ndcg5_large,ndcg10_large
0,Híbrido (CTR + embeddings),0.6832,0.3349,0.3712,0.4333,0.6828,0.3355,0.3721,0.4340
1,Baseline: CTR reciente 24 h,0.6526,0.3115,0.3403,0.4025,0.6534,0.3129,0.3418,0.4038
2,Embeddings MiniLM,0.6407,0.3084,0.3370,0.3966,0.6401,0.3085,0.3369,0.3969
3,Azar,0.5007,0.2188,0.2238,0.2866,0.4993,0.2184,0.2229,0.2858


In [18]:
print(tiempos.to_string(index=False))
print(f"\nTiempo del script en EMR: {tiempos['segundos'].max() / 60:.1f} min "
      f"(el step completo, con arranque de Spark, tardó 3 h 44 min)")

                                                                      etapa  segundos
                                                                p0 = 0.0405        20
                                              Candidatas de dev: 14,085,557     13334
                                             Impresiones evaluadas: 376,471     13421
aleatorio  AUC = 0.4993 | MRR = 0.2184 | nDCG@5 = 0.2229 | nDCG@10 = 0.2858     13428
ctr        AUC = 0.6534 | MRR = 0.3129 | nDCG@5 = 0.3418 | nDCG@10 = 0.4038     13428
emb_resp   AUC = 0.6401 | MRR = 0.3085 | nDCG@5 = 0.3369 | nDCG@10 = 0.3969     13428
hibrido    AUC = 0.6828 | MRR = 0.3355 | nDCG@5 = 0.3721 | nDCG@10 = 0.4340     13428
                                                       Resultados guardados     13435

Tiempo del script en EMR: 223.9 min (el step completo, con arranque de Spark, tardó 3 h 44 min)


In [19]:
filas = []
for met in ("auc", "mrr", "ndcg5", "ndcg10"):
    dif, ic, medias = paired_bootstrap(m[f"{met}_hibrido"], m[f"{met}_ctr"])
    filas.append({"metrica": met, "diferencia": dif, "ic_inf": ic[0], "ic_sup": ic[1],
                  "p_dif_menor_igual_0.01": np.mean(medias <= 0.01)})
bootstrap_large = pd.DataFrame(filas)

auc = bootstrap_large.iloc[0]
print(f"H1 en MIND-large (AUC híbrido − AUC baseline ≥ 0.01): "
      f"{'SE ACEPTA' if auc['ic_inf'] > 0.01 else 'NO SE ACEPTA'} "
      f"(IC 95 % = [{auc['ic_inf']:+.4f}, {auc['ic_sup']:+.4f}])")
bootstrap_large.round(4)

H1 en MIND-large (AUC híbrido − AUC baseline ≥ 0.01): SE ACEPTA (IC 95 % = [+0.0288, +0.0299])


,metrica,diferencia,ic_inf,ic_sup,p_dif_menor_igual_0.01
0,auc,0.0294,0.0288,0.0299,0.0
1,mrr,0.0226,0.0220,0.0231,0.0
2,ndcg5,0.0303,0.0297,0.0309,0.0
3,ndcg10,0.0302,0.0296,0.0307,0.0


In [20]:
resumen.assign(dataset="large", periodo="dev").to_csv(RES / "08_large_resumen.csv", index=False)
comp.to_csv(RES / "08_small_vs_large.csv", index=False)
bootstrap_large.to_csv(RES / "08_large_bootstrap.csv", index=False)
tiempos.to_csv(RES / "08_large_tiempos.csv", index=False)
print("Guardado en results/metrics/")

Guardado en results/metrics/
